## add band using dates

In [18]:
import rasterio, glob
import numpy as np

# Input GeoTIFF file path
for input_tiff in sorted(glob.glob('../output/flood_raster/reprojected/*extent*148*.tif')):
    print(f'Processing {input_tiff}...')

    # Output GeoTIFF file path for the stacked bands
    output_tiff = input_tiff.replace('/reprojected/', '/monthlyadded/').replace('.tif', '_monthly.tif')

    # Open the input GeoTIFF file to get width, height, and metadata
    with rasterio.open(input_tiff) as src:
        width = src.width
        height = src.height
        metadata = src.meta.copy()
        bandnames = src.descriptions

        # Extract unique months from band names
        band_name_patterns = list(set([bandname[:6] for bandname in bandnames]))

        if bandnames[0] != None:
            # Initialize an empty array to store the stacked bands
            stacked_bands = []

            # Process each wildcard pattern
            for band_n, band_name_pattern in enumerate(band_name_patterns):
                # Get band names that match the current wildcard pattern
                selected_band_indices = [i for i, name in enumerate(bandnames, start=1) if band_name_pattern in name]

                # Read selected bands and create a sum for the current pattern
                combined_band = np.zeros((height, width), dtype=np.uint8)  # Assuming dtype is uint8
                for band_index in selected_band_indices:
                    with rasterio.open(input_tiff) as src:
                        band_arr = src.read(band_index)
                        combined_band += (band_arr == 3).astype(np.uint8)  # Assuming condition for summing is (band_arr == 3)

                # Append the combined band to the list
                stacked_bands.append(combined_band)

            # Stack the bands along the third axis to create a 3-band array
            stacked_array = np.stack(stacked_bands, axis=2)

            # Update metadata for the output file (number of bands and data type)
            metadata.update({
                'count': len(band_name_patterns),  # Number of bands in the output file
                'dtype': np.uint8,  # Data type of the stacked bands (uint8)
                'nodata': 255  # Set the nodata value within the valid range of uint8 (0 to 255)
            })

            # Write the stacked bands to the output GeoTIFF file
            with rasterio.open(output_tiff, 'w', **metadata) as dst:
                # Write the stacked array as bands in the output file
                for i, band_name in enumerate(band_name_patterns):
                    dst.write(stacked_array[:, :, i], i+1)  # Write each band separately
                    dst.set_band_description(i+1, band_name)  # Set band names


Processing ../output/flood_raster/reprojected/floodextentstacked_2018_2022_148_reprojected.tif...


## compress the folder so that it is easier to download

In [24]:
import zipfile
import os

def zip_folder(folder_path, zip_filename):
    # Create a Zip file
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Walk through all the files in the folder
        for root, _, files in os.walk(folder_path):
            for file in files:
                # Create the full file path
                file_path = os.path.join(root, file)
                # Add the file to the Zip file
                zipf.write(file_path, os.path.relpath(file_path, folder_path))

# Example usage
folder_to_zip = '../output/flood_raster/monthlyadded/'  # Replace this with the path to your folder
zip_file_name = '../output/flood_raster/flood_raster_monthlyadded.zip'  # Name for the zip file

zip_folder(folder_to_zip, zip_file_name)
print(f'Folder "{folder_to_zip}" has been compressed to "{zip_file_name}"')

Folder "../output/flood_raster/monthlyadded/" has been compressed to "../output/flood_raster/flood_raster_monthlyadded.zip"
